In [3]:
from pathlib import Path

In [4]:
Path.cwd()

PosixPath('/home/hack/Desktop/freelancer/qrCode_3')

In [5]:
root = Path.cwd()
exclude_dirs = {"__pycache__", "logs", ".git", "core", "alembic"}
exclude_files = {
    "README.md",
    "t.ipynb",
    ".dockerignore",
    ".gitignore",
    "asset.jpeg",
    "run.sh",
    "init.sh",
    "alembic.init",
}

result = []

for path in root.rglob("*"):  # recursive
    if path.is_file():
        # Skip excluded directories
        if any(part in exclude_dirs for part in path.parts):
            continue

        # Skip excluded file names
        if path.name in exclude_files:
            continue

        result.append(path)

In [6]:
result = [i.relative_to(root) for i in result]

In [7]:
s = ""
for i in result:
    s += f"```{str(i)}\n"
    s += i.read_text(encoding="utf-8")
    s += "```\n\n"

In [8]:
print(s)

```Dockerfile
FROM python:3.13-slim

WORKDIR /app

RUN apt-get update && apt-get install -y vim \
    gcc \
    build-essential \
    python3-dev \
    fonts-liberation

COPY ./app .

RUN pip install -r requirements.txt


CMD ["/bin/sh", "-c", "alembic revision --autogenerate -m 'Start' && alembic upgrade head && uvicorn main:app --host 0.0.0.0 --port ${PORT:-8000}"]```

```app/main.py
from fastapi import FastAPI, Depends
from contextlib import asynccontextmanager
from src.core.logging_config import get_logger
from fastapi.middleware.cors import CORSMiddleware
from src.core.config import settings
from src.auth.router import router as auth_router
from fastapi.responses import FileResponse
from src.auth.service import check_in_by_qr_code
from sqlalchemy.orm import Session
from src.core.session import get_db

# Setup logging
logger = get_logger()


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("App starting...")
    yield
    print("App shutting down...")


app = FastAP